# Table of contents

* **Introduction**
* **Objective**
* **Pipeline Overview**
* **Data Dictionary**
* **Approach**
* **Name Entity Recognition Implementation**
    * Setup Environments
    * Data Loading
    * Data Visualization
    * Text Normalization
    * Named Entity Recognition (NLTK)
        * NER Implementation (NLTK)
        * NER Results Visualization (NLTK)
    * Named Entity Recognition (spaCy)
        * NER Implementation (spaCy)
        * NER Results Visualization (spaCy)
    * NLTK vs. spaCy Comparison
* **Conclusion**
* **References**

# Introduction

Named Entity Recognition (NER) is a core task in natural language processing that identifies and classifies named entities (such as organizations, locations, persons, products, cardinal numbers, etc.) in unstructured text. This implementation applies two popular NER approaches — the traditional NLTK-based chunker (using maximum entropy classifier and gazetteer) and the modern statistical model from spaCy (small English model) — to a small sample of 3 arXiv paper abstracts from computer science domains. The code demonstrates entity extraction, detailed comparison of results between both libraries, and visual highlighting using spaCy's built-in displaCy renderer for both entity and dependency parsing views. This serves as an educational example of how different NER systems perform on dense, technical scientific text.

# Objective

The primary goal is to perform a comparative analysis of NER capabilities between NLTK’s rule-based/statistical chunking and spaCy’s modern statistical models. The study aims to highlight differences in entity granularity, accuracy in technical contexts, and the quality of linguistic visualization.

# Pipeline Overview

The pipeline is designed to be lightweight and iterative, focusing on clear comparisons:

- Environment Setup: Installation of textacy, spaCy, and specific NLTK datasets (punkt, POS taggers).
- Data Sampling: Loading and selecting a small sample (3 abstracts) from the arXiv dataset for rapid demonstration.
- Preprocessing: Tokenization and POS tagging to create a baseline for extraction.
- NLTK-based Named Entity Recognition:
    - Apply NLTK's `ne_chunk` on POS-tagged sentences then extract entity spans and types (PERSON, ORGANIZATION, GPE, LOCATION, etc.).
    - Visualize NLTK tagging in each paragraph
- spaCy-based Named Entity Recognition
    - Reconstruct full text from tokens to process with spaCy `en_core_web_sm` then extract entities (ORG, PRODUCT, GPE, CARDINAL, NORP, etc.).
    - Render entity highlighting (colored spans) for each sample using displaCy (`style="ent"`).
    - Render dependency parse trees (`style="dep"`) to provide syntactic context around entities.
- Result Comparison:
    - Print side-by-side table of entities detected by NLTK vs spaCy for each sample.
    - Show quantitative and qualitative differences.

# Data Dictionary

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-za14{border-color:inherit;text-align:left;vertical-align:bottom}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-za14">Variable Name</th>
    <th class="tg-7zrl">Description</th>
    <th class="tg-7zrl">Data Type</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-7zrl">titles</td>
    <td class="tg-7zrl">The title of the arXiv research paper. Includes approximately 39,000 unique values across the dataset.</td>
    <td class="tg-7zrl">Text/String</td>
  </tr>
  <tr>
    <td class="tg-7zrl">summaries</td>
    <td class="tg-7zrl">The abstract or summary of the arXiv paper. These provide a concise overview of the research and findings.</td>
    <td class="tg-7zrl">Text/String</td>
  </tr>
  <tr>
    <td class="tg-7zrl">terms</td>
    <td class="tg-7zrl">Categorical tags representing the arXiv research areas (e.g., Computer Vision, Machine Learning).</td>
    <td class="tg-7zrl">List/Categorical</td>
  </tr>
</tbody>
</table>

[**Dataset Link**](https://www.kaggle.com/datasets/spsayakpaul/arxiv-paper-abstracts)

# Approach

This implementation follows a comparative, visualization-heavy methodology using two complementary NER systems:

**Data Preparation Approach**

A minimal sample of 3 abstracts is used to keep execution fast and output readable. Text is tokenized and POS-tagged once (with NLTK) to serve both NER pipelines, ensuring fair comparison from the same base tokens.

**NLTK NER Approach**

Uses the classic maximum-entropy chunker (ne_chunk) trained on CoNLL-2003 data. It operates directly on POS-tagged token lists and produces coarse-grained entity types (mainly PERSON, ORGANIZATION, GPE, LOCATION). This method is lightweight but tends to over-generalize acronyms and technical terms as ORGANIZATION.

**spaCy NER Approach**

Uses the small English statistical model (en_core_web_sm), which is fast and reasonably accurate for general domains. It processes raw text (reconstructed from tokens) and recognizes a richer set of labels (including CARDINAL, PRODUCT, NORP, ORG, etc.). spaCy generally handles acronyms, numbers, and technical phrases more precisely.

**Comparison & Visualization Strategy**

- Entities from both systems are collected and displayed side-by-side in a simple text table.
- spaCy’s displaCy is used for high-quality, color-coded entity visualization (style="ent") and dependency tree rendering (style="dep") to provide deeper linguistic context.
- The focus is on qualitative differences: spaCy tends to detect more numeric entities (CARDINAL) and treats model/framework names as PRODUCT or ORG more reliably, while NLTK often labels them as ORGANIZATION or misses them.

# Name Entity Recognition Implementation

## Setup Environments

**Packages Installation**

Quietly installs textacy and downloads the small English spaCy model (`en_core_web_sm`) required for advanced linguistic processing.

In [1]:
%%capture
!pip install -q textacy
!python -m spacy download en_core_web_sm -q

**Import Libraries**

Imports key libraries (`pandas, spaCy, textacy, NLTK, NetworkX, matplotlib, displaCy`). Downloads essential NLTK resources (`punkt, words, maxent_ne_chunker, averaged_perceptron_tagger`). Configures pandas to display full text and loads the spaCy English model.

In [ ]:
import pandas as pd
import spacy
from spacy import displacy
import textacy
from textacy.extract import keyterms
import nltk
from nltk import ne_chunk, pos_tag
import networkx as nx
import matplotlib.pyplot as plt

nltk.download('punkt')
nltk.download('words')
nltk.download('maxent_ne_chunker_tab')
nltk.download('averaged_perceptron_tagger_eng')

pd.set_option('display.max_colwidth', None)

nlp = spacy.load("en_core_web_sm")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package words to /usr/share/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


## Data Loading

Loads the arXiv abstracts dataset from CSV. Randomly samples only 3 abstracts (fixed seed) for quick demostration for this tutorial notebook, focused demonstration and keeps only the summaries column.

In [3]:
df = pd.read_csv('/kaggle/input/arxiv-paper-abstracts/arxiv_data.csv')
df = df.sample(n=3, random_state=42)
df

,titles,summaries,terms
29096,Enforcing geometric constraints of virtual normal for depth prediction,"Monocular depth prediction plays a crucial role in understanding 3D scene\ngeometry. Although recent methods have achieved impressive progress in\nevaluation metrics such as the pixel-wise relative error, most methods neglect\nthe geometric constraints in the 3D space. In this work, we show the importance\nof the high-order 3D geometric constraints for depth prediction. By designing a\nloss term that enforces one simple type of geometric constraints, namely,\nvirtual normal directions determined by randomly sampled three points in the\nreconstructed 3D space, we can considerably improve the depth prediction\naccuracy. Significantly, the byproduct of this predicted depth being\nsufficiently accurate is that we are now able to recover good 3D structures of\nthe scene such as the point cloud and surface normal directly from the depth,\neliminating the necessity of training new sub-models as was previously done.\nExperiments on two benchmarks: NYU Depth-V2 and KITTI demonstrate the\neffectiveness of our method and state-of-the-art performance.",['cs.CV']
2638,Chart Auto-Encoders for Manifold Structured Data,"Deep generative models have made tremendous advances in image and signal\nrepresentation learning and generation. These models employ the full Euclidean\nspace or a bounded subset as the latent space, whose flat geometry, however, is\noften too simplistic to meaningfully reflect the manifold structure of the\ndata. In this work, we advocate the use of a multi-chart latent space for\nbetter data representation. Inspired by differential geometry, we propose a\n\textbf{Chart Auto-Encoder (CAE)} and prove a universal approximation theorem\non its representation capability. We show that the training data size and the\nnetwork size scale exponentially in approximation error with an exponent\ndepending on the intrinsic dimension of the data manifold. CAE admits desirable\nmanifold properties that auto-encoders with a flat latent space fail to obey,\npredominantly proximity of data. We conduct extensive experimentation with\nsynthetic and real-life examples to demonstrate that CAE provides\nreconstruction with high fidelity, preserves proximity in the latent space, and\ngenerates new data remaining near the manifold. These experiments show that CAE\nis advantageous over existing auto-encoders and variants by preserving the\ntopology of the data manifold as well as its geometry.","['cs.LG', 'stat.ML']"
28705,SASO: Joint 3D Semantic-Instance Segmentation via Multi-scale Semantic Association and Salient Point Clustering Optimization,"We propose a novel 3D point cloud segmentation framework named SASO, which\njointly performs semantic and instance segmentation tasks. For semantic\nsegmentation task, inspired by the inherent correlation among objects in\nspatial context, we propose a Multi-scale Semantic Association (MSA) module to\nexplore the constructive effects of the semantic context information. For\ninstance segmentation task, different from previous works that utilize\nclustering only in inference procedure, we propose a Salient Point Clustering\nOptimization (SPCO) module to introduce a clustering procedure into the\ntraining process and impel the network focusing on points that are difficult to\nbe distinguished. In addition, because of the inherent structures of indoor\nscenes, the imbalance problem of the category distribution is rarely considered\nbut severely limits the performance of 3D scene perception. To address this\nissue, we introduce an adaptive Water Filling Sampling (WFS) algorithm to\nbalance the category distribution of training data. Extensive experiments\ndemonstrate that our method outperforms the state-of-the-art methods on\nbenchmark datasets in both semantic segmentation and instance segmentation\ntasks.","['cs.CV', 'cs.RO']"


**Initial Data Preprocessing**

Select only necessary column to do NER then prints the full text of the 3 selected abstracts to provide insight into the scientific content and style.

In [4]:
df = df["summaries"]
df

29096                                                                                                                                                                                                                                               Monocular depth prediction plays a crucial role in understanding 3D scene\ngeometry. Although recent methods have achieved impressive progress in\nevaluation metrics such as the pixel-wise relative error, most methods neglect\nthe geometric constraints in the 3D space. In this work, we show the importance\nof the high-order 3D geometric constraints for depth prediction. By designing a\nloss term that enforces one simple type of geometric constraints, namely,\nvirtual normal directions determined by randomly sampled three points in the\nreconstructed 3D space, we can considerably improve the depth prediction\naccuracy. Significantly, the byproduct of this predicted depth being\nsufficiently accurate is that we are now able to recover good 3D struc

## Data Visualization

Display texts in the corpus.

In [ ]:
random_rows = df

for idx, content in random_rows.items():
    print(f"Row Index: {idx}")
    print(content)
    print("-" * 50) 

Row Index: 29096
Monocular depth prediction plays a crucial role in understanding 3D scene
geometry. Although recent methods have achieved impressive progress in
evaluation metrics such as the pixel-wise relative error, most methods neglect
the geometric constraints in the 3D space. In this work, we show the importance
of the high-order 3D geometric constraints for depth prediction. By designing a
loss term that enforces one simple type of geometric constraints, namely,
virtual normal directions determined by randomly sampled three points in the
reconstructed 3D space, we can considerably improve the depth prediction
accuracy. Significantly, the byproduct of this predicted depth being
sufficiently accurate is that we are now able to recover good 3D structures of
the scene such as the point cloud and surface normal directly from the depth,
eliminating the necessity of training new sub-models as was previously done.
Experiments on two benchmarks: NYU Depth-V2 and KITTI demonstrate the
ef

Calculates word count for each abstract. Finds and prints the longest abstract to understand the maximum text length in the sample.

In [ ]:
word_counts = df.str.split().str.len()
longest_idx = word_counts.idxmax()

max_word_count = word_counts.max()

longest_paragraph = df.loc[longest_idx]

print(f"Row Index: {longest_idx}")
print(f"Word Count: {max_word_count}")
print("-" * 30)
print("Full Paragraph:")
print(longest_paragraph)

Row Index: 2638
Word Count: 187
------------------------------
Full Paragraph:
Deep generative models have made tremendous advances in image and signal
representation learning and generation. These models employ the full Euclidean
space or a bounded subset as the latent space, whose flat geometry, however, is
often too simplistic to meaningfully reflect the manifold structure of the
data. In this work, we advocate the use of a multi-chart latent space for
better data representation. Inspired by differential geometry, we propose a
\textbf{Chart Auto-Encoder (CAE)} and prove a universal approximation theorem
on its representation capability. We show that the training data size and the
network size scale exponentially in approximation error with an exponent
depending on the intrinsic dimension of the data manifold. CAE admits desirable
manifold properties that auto-encoders with a flat latent space fail to obey,
predominantly proximity of data. We conduct extensive experimentation with
sy

## Text Normalization

Sentence segmentation splits each abstract into individual sentences using NLTK `sent_tokenize`.

In [ ]:
df_segmented = df.apply(nltk.tokenize.sent_tokenize)

print("Segmented Data (First 5 rows):")
print(df_segmented.head())

Segmented Data (First 5 rows):
29096                                                                                                                                                                                                                                                  [Monocular depth prediction plays a crucial role in understanding 3D scene\ngeometry., Although recent methods have achieved impressive progress in\nevaluation metrics such as the pixel-wise relative error, most methods neglect\nthe geometric constraints in the 3D space., In this work, we show the importance\nof the high-order 3D geometric constraints for depth prediction., By designing a\nloss term that enforces one simple type of geometric constraints, namely,\nvirtual normal directions determined by randomly sampled three points in the\nreconstructed 3D space, we can considerably improve the depth prediction\naccuracy., Significantly, the byproduct of this predicted depth being\nsufficiently accurate is that w

In [8]:
print("\nSentences in Row:")
for i, sentence in enumerate(df_segmented.iloc[0]):
    print(f"{i}: {sentence}")


Sentences in Row:
0: Monocular depth prediction plays a crucial role in understanding 3D scene
geometry.
1: Although recent methods have achieved impressive progress in
evaluation metrics such as the pixel-wise relative error, most methods neglect
the geometric constraints in the 3D space.
2: In this work, we show the importance
of the high-order 3D geometric constraints for depth prediction.
3: By designing a
loss term that enforces one simple type of geometric constraints, namely,
virtual normal directions determined by randomly sampled three points in the
reconstructed 3D space, we can considerably improve the depth prediction
accuracy.
4: Significantly, the byproduct of this predicted depth being
sufficiently accurate is that we are now able to recover good 3D structures of
the scene such as the point cloud and surface normal directly from the depth,
eliminating the necessity of training new sub-models as was previously done.
5: Experiments on two benchmarks: NYU Depth-V2 and KITT

Word tokenization breaks sentences into lists of words/tokens.

In [ ]:
df_tokenized = df_segmented.apply(lambda sentences: [nltk.tokenize.word_tokenize(s) for s in sentences])

print("\nWord Tokenized Data (First 5 rows):")
print(df_tokenized.head())


Word Tokenized Data (First 5 rows):
29096                                                                                                                                                                                                                                                                                               [[Monocular, depth, prediction, plays, a, crucial, role, in, understanding, 3D, scene, geometry, .], [Although, recent, methods, have, achieved, impressive, progress, in, evaluation, metrics, such, as, the, pixel-wise, relative, error, ,, most, methods, neglect, the, geometric, constraints, in, the, 3D, space, .], [In, this, work, ,, we, show, the, importance, of, the, high-order, 3D, geometric, constraints, for, depth, prediction, .], [By, designing, a, loss, term, that, enforces, one, simple, type, of, geometric, constraints, ,, namely, ,, virtual, normal, directions, determined, by, randomly, sampled, three, points, in, the, reconstructed, 3D, space, ,, we, ca

In [10]:
# View the specific word tokens in the first sentence of the first row
print("\nWords in Row 0, Sentence 0:")
print(df_tokenized.iloc[0][0])


Words in Row 0, Sentence 0:
['Monocular', 'depth', 'prediction', 'plays', 'a', 'crucial', 'role', 'in', 'understanding', '3D', 'scene', 'geometry', '.']


## Named Entity Recognition (NLTK)

In Natural Language Processing (NLP), Part-of-Speech (POS) Tagging is the process of labeling each word in a text with its corresponding grammatical category (such as noun, verb, adjective, etc.). When using the Natural Language Toolkit (NLTK) for Named Entity Recognition (NER), POS tagging acts as the essential "feature engineering" step that allows the computer to identify which words are likely to be names, places, or organizations.

**The Role of POS Tagging in NER**

Named Entity Recognition relies heavily on Proper Nouns. Without POS tagging, an algorithm cannot easily distinguish between:

- "I want to apple the paint" (Verb - not an entity)
- "I bought an apple" (Common Noun - usually not a named entity)
- "I work for Apple" (Proper Noun - a Named Entity)

NLTK uses POS tags to narrow the search space for its NER classifier, focusing primarily on tokens tagged as NNP (Proper Noun, Singular) or NNPS (Proper Noun, Plural).

**Foundations**

**A. The Penn Treebank Tagset**

NLTK primarily uses the Penn Treebank tagset. For NER, the most critical tags are:
- NNP: Proper Noun, Singular (e.g., "London", "Elon")
- NNPS: Proper Noun, Plural (e.g., "Americans")
- JJ: Adjective (often part of an entity, like "British" in "British Airways")

**B. Pointwise Prediction**

NLTK’s default pos_tag() uses a Perceptron Tagger. It looks at the current word and its immediate context (the words before and after) to predict the tag. This is crucial for NER because the same word can change roles:

- "I saw Bill" (Proper Noun/Entity)
- "Send me the bill" (Common Noun/Non-entity)

**C. N-Gram Tagging Theory**

NLTK allows for "N-gram" taggers (Unigram, Bigram, Trigram). These work on the principle of Markov Chains:

- A Bigram Tagger calculates the probability of a tag based on the tag of the previous word.
- In NER, if the previous tag was a "Title" (like "Mr."), there is a high probability the next word is an NNP (Proper Noun).

**Connecting POS Tagging to NER**

Once NLTK has tagged your text, NER is performed using two main conceptual approaches:

**A. Rule-Based Chunking (Regex)**

You define patterns based on POS tags to "chunk" words together.

- Concept: "An entity is any sequence of one or more Proper Nouns (NNP)."
- Pattern: {(<NNP>)+}

**B. IOB Labeling (Inside, Outside, Beginning)**

To represent entities in a sequence, NLTK uses the IOB format. This theory solves the "multi-word entity" problem:

- B-PERSON: Beginning of a person's name.
- I-PERSON: Inside/continuation of the name.
- O: Outside any named entity.

**Key Implementation Steps**

In NLTK, the process is typically executed with the following logic:
- nltk.pos_tag(tokens): Takes a list of words and returns a list of tuples (Word, Tag).
- nltk.ne_chunk(tagged_tokens): Uses a pre-trained classifier to identify "PERSON", "ORGANIZATION", and "GPE" based on those tags.
- Binary NER: You can set binary=True in the chunker if you only care if something is an entity, regardless of whether it's a person or a place.

**Summary Table: POS Tags most used by NER**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-za14{border-color:inherit;text-align:left;vertical-align:bottom}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-za14">POS Tag</th>
    <th class="tg-za14">Meaning</th>
    <th class="tg-za14">Entity Example</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-za14">NNP</td>
    <td class="tg-za14">Proper Noun</td>
    <td class="tg-0pky">"Einstein" (PERSON)</td>
  </tr>
  <tr>
    <td class="tg-za14">NNPS</td>
    <td class="tg-za14">Proper Noun, Plural</td>
    <td class="tg-0pky">"Vikings" (ORGANIZATION)</td>
  </tr>
  <tr>
    <td class="tg-za14">GPE</td>
    <td class="tg-za14">Geo-Political Entity</td>
    <td class="tg-0pky">"Germany" (LOCATION)</td>
  </tr>
</tbody>
</table>

### NER Implementation (NLTK)

Runs NLTK `pos_tag` on all tokenized sentences to assign part-of-speech tags to each word. 

In [ ]:
df_pos_tagged = df_tokenized.apply(lambda sentences: [pos_tag(s) for s in sentences])

def extract_nltk_entities(pos_sentences):
    entities = []
    for sentence in pos_sentences:
        chunks = ne_chunk(sentence)
        for chunk in chunks:
            if hasattr(chunk, 'label'):
                entity_name = " ".join(c[0] for c in chunk)
                entity_type = chunk.label()
                entities.append((entity_name, entity_type))
    return entities

df_entities = df_pos_tagged.apply(extract_nltk_entities)

print("\nNER Results:")
print(df_entities.head())


NER Results:
29096                                                                                                                      [(NYU, ORGANIZATION), (KITTI, ORGANIZATION)]
2638                          [(Deep, GPE), (Euclidean, LOCATION), (Chart, PERSON), (CAE, ORGANIZATION), (CAE, ORGANIZATION), (CAE, ORGANIZATION), (CAE, ORGANIZATION)]
28705    [(SASO, ORGANIZATION), (MSA, ORGANIZATION), (Salient Point Clustering Optimization, ORGANIZATION), (SPCO, ORGANIZATION), (Sampling, GPE), (WFS, ORGANIZATION)]
Name: summaries, dtype: object


### NER Results Visualization (NLTK)

Displays each abstract's full original text together with the list of entities found by NLTK in a clean, formatted way.

In [ ]:
print("\n--- Detailed NLTK NER Breakdown ---")

for i, (idx, entities) in enumerate(df_entities.head(3).items(), 1):
    print(f"\n{'='*30} Row {i} (Index: {idx}) {'='*30}")
    
    full_paragraph = df.loc[idx] 
    print(f"FULL TEXT:\n{full_paragraph}")
    
    print(f"\nEXTRACTED ENTITIES:")
    if not entities:
        print("  [No entities found]")
    else:
        for name, etype in entities:
            print(f"  - {name:<25} [{etype}]")


--- Detailed NLTK NER Breakdown ---

============================== Row 1 (Index: 29096) ==============================
FULL TEXT:
Monocular depth prediction plays a crucial role in understanding 3D scene
geometry. Although recent methods have achieved impressive progress in
evaluation metrics such as the pixel-wise relative error, most methods neglect
the geometric constraints in the 3D space. In this work, we show the importance
of the high-order 3D geometric constraints for depth prediction. By designing a
loss term that enforces one simple type of geometric constraints, namely,
virtual normal directions determined by randomly sampled three points in the
reconstructed 3D space, we can considerably improve the depth prediction
accuracy. Significantly, the byproduct of this predicted depth being
sufficiently accurate is that we are now able to recover good 3D structures of
the scene such as the point cloud and surface normal directly from the depth,
eliminating the necessity of train

## Named Entity Recognition (spaCy)

spaCy is an industry-standard, open-source library for advanced Natural Language Processing (NLP) in Python. In the context of Named Entity Recognition (NER), it is widely considered the most "production-ready" tool because it balances high-speed processing with state-of-the-art accuracy.

While NLTK (which you used previously) is excellent for education and basic rule-based tasks, spaCy uses sophisticated statistical models (like Convolutional Neural Networks and Transformers) to "understand" context.

**The Core Technology: Statistical Models**

The fundamental difference with spaCy is that it uses pre-trained statistical models (like en_core_web_sm) to predict entities.
- Transition-Based System: Rather than just looking at a word's tag, spaCy uses a "transition-based" approach. It processes the text word-by-word and uses a neural network to decide whether to start an entity, continue an entity, or end one.
- Contextual Word Embeddings: spaCy models use multi-dimensional vectors (word embeddings) to understand that "Apple" the fruit and "Apple" the company exist in different "vector spaces" based on the words surrounding them.

**How spaCy NER Works**

Unlike simple keyword matching, spaCy looks at the entire sentence structure to decide what a word is. For example, it can distinguish between "Apple" as a company (ORG) and "apple" as a fruit (not an entity) based on the surrounding tokens and capitalization.

**A. The Doc Object and the Pipeline**

When you process text in spaCy, it passes through a pipeline. The NER component is typically the final stage:

- Tagger: Assigns POS tags (Noun, Verb).
- Parser: Understands the grammatical structure (Dependency parsing).
- NER: Labels the spans of text.

**B. Entity Labels (The Schema)**

spaCy uses a much more detailed set of labels than the standard NLTK chunker. Common labels include:

- ORG: Companies, agencies, institutions.
- GPE: Countries, cities, states.
- FAC: Buildings, airports, highways.
- NORP: Nationalities or religious/political groups.
- MONEY: Monetary values, including units.

**C. The BILUO Scheme**

To achieve high precision, spaCy uses the BILUO tagging scheme (more detailed than NLTK's IOB):

- B (Begin): The first token of a multi-token entity.
- I (In): An inner token of a multi-token entity.
- L (Last): The final token of a multi-token entity.
- U (Unit): A single-token entity.
- O (Out): Not part of an entity.

**Entity Labels**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-za14{border-color:inherit;text-align:left;vertical-align:bottom}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-za14">Label</th>
    <th class="tg-za14">Description</th>
    <th class="tg-za14">Example</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-za14">PERSON</td>
    <td class="tg-za14">People, including fictional characters.</td>
    <td class="tg-0pky">Elon Musk, Harry Potter</td>
  </tr>
  <tr>
    <td class="tg-za14">ORG</td>
    <td class="tg-za14">Companies, agencies, or institutions.</td>
    <td class="tg-0pky">Google, NASA, UN</td>
  </tr>
  <tr>
    <td class="tg-za14">GPE</td>
    <td class="tg-za14">Geopolitical entities (Countries, Cities).</td>
    <td class="tg-0pky">Tokyo, Germany, California</td>
  </tr>
  <tr>
    <td class="tg-7zrl">DATE</td>
    <td class="tg-7zrl">Absolute or relative dates/periods.</td>
    <td class="tg-0lax">2024, yesterday, three weeks</td>
  </tr>
  <tr>
    <td class="tg-7zrl">MONEY</td>
    <td class="tg-7zrl">Monetary values, including units.</td>
    <td class="tg-0lax">$10 million, 500 Euro</td>
  </tr>
  <tr>
    <td class="tg-7zrl">PRODUCT</td>
    <td class="tg-7zrl">Objects, vehicles, foods, etc.</td>
    <td class="tg-0lax">iPhone, Tesla Model S</td>
  </tr>
</tbody></table>

### NER Implementation (spaCy)

Reconstructs the full text from tokens. Processes it with spaCy en_core_web_sm to detect entities (richer tag set: ORG, PRODUCT, CARDINAL, NORP, GPE, etc.). Stores results.

In [ ]:
df_sample = df_tokenized.to_frame(name='tokenized_content')

print(f"Rendering NER Visualizations...\n")

for i, (idx, row) in enumerate(df_sample.iterrows(), 1):
    full_text = " ".join([" ".join(sentence) for sentence in row['tokenized_content']])
    
    doc = nlp(full_text)
    
    print(f"--- Sample {i} (Original Index: {idx}) ---")
    displacy.render(doc, style="ent", jupyter=True)
    print("\n" + "="*80 + "\n")

Rendering 5 NER Visualizations...

--- Sample 1 (Original Index: 29096) ---




--- Sample 2 (Original Index: 2638) ---




--- Sample 3 (Original Index: 28705) ---


### NER Results Visualization (spaCy)

**Detailed spaCy NER Output**

Prints the reconstructed full paragraph and complete list of spaCy-detected entities for each sample.

**spaCy Entity Visualization**

Uses displaCy to render beautiful, color-coded entity highlighting (style="ent") for all 3 abstracts.

**spaCy Dependency Parsing Visualization**

Renders syntactic dependency trees (style="dep") with dark theme styling for each abstract, showing grammatical structure around entities.

In [ ]:
options = {
    "distance": 100, 
    "compact": True, 
    "color": "#FFFFFF", 
    "bg": "#2b2b2b",
    "font": "Arial"
}

num_samples = 3

print(f"--- Displaying Full Text and Dependency Graphs ---")

for i in range(num_samples):
    sample_tokens = df_sample.iloc[i]['tokenized_content']
    current_idx = df_sample.index[i]
    
    full_text = " ".join([" ".join(sentence) for sentence in sample_tokens])
    
    print(f"\n{'='*100}")
    print(f"SAMPLE {i+1} (Index: {current_idx})")
    print(f"{'='*100}")
    print(f"FULL TEXT")
    
    doc = nlp(full_text)
    displacy.render(doc, style="ent", jupyter=True)
    print(f"DEPENDENCY GRAPH:")
    displacy.render(doc, style="dep", jupyter=True, options=options)

--- Displaying Full Text and Dependency Graphs ---

SAMPLE 1 (Index: 29096)
FULL TEXT


DEPENDENCY GRAPH:



SAMPLE 2 (Index: 2638)
FULL TEXT


DEPENDENCY GRAPH:



SAMPLE 3 (Index: 28705)
FULL TEXT


DEPENDENCY GRAPH:


## NLTK vs. spaCy Comparison

When choosing between spaCy and NLTK for Named Entity Recognition (NER), the decision usually comes down to whether you are in an academic/research setting or a production/software development environment.

**Foundations Differences**

**The Pipeline Philosophy**

NLTK follows a "bottom-up" modular approach. You must manually tokenize the text, then tag the parts of speech, and then pass that specific data into a chunker. This gives you total control over every step, which is great for learning the theory of linguistics.

spaCy follows an "integrated pipeline" approach. When you load a model and process a string, the text passes through a pre-defined neural network. The NER component automatically benefits from the POS tagger and the Dependency Parser running simultaneously in the background.

**Algorithm Architecture**

NLTK generally uses a Maximum Entropy (MaxEnt) classifier for its ne_chunk function. It looks at local features (like "Is the first letter capitalized?") to make a prediction.

spaCy uses a Transition-Based system powered by a CNN (Convolutional Neural Network) or Transformers. It views the sentence as a series of states and uses multi-dimensional word embeddings to understand context more deeply than NLTK can.

**Tagging Schemes: IOB vs. BILUO**

One of the major technical differences is how they handle multi-word entities (e.g., "The New York Times").

- NLTK (IOB): Uses Inside, Outside, Beginning. It only knows if a word starts an entity or is inside one.
- spaCy (BILUO): Uses Begin, Inside, Last, Unit, Outside. By having a specific tag for the "Last" word of a phrase and a "Unit" tag for single-word names, spaCy is much better at identifying where a long name actually ends.

**When to Use Which?**

Use NLTK if:
- You are learning NLP and want to understand the individual steps of the process.
- You need to build a custom grammar from scratch for a very specific, niche language.
- You are performing a task that requires very simple rule-based patterns.

Use spaCy if:
- You are building a real-world application (App, Bot, or API).
- You need to process large volumes of data quickly.
- Accuracy is your top priority, especially for complex entities like product names or dates.
- You want built-in support for Transformers (BERT/RoBERTa).

**High-Level Comparison Table**

<style type="text/css">
.tg  {border-collapse:collapse;border-spacing:0;}
.tg td{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  overflow:hidden;padding:10px 5px;word-break:normal;}
.tg th{border-color:black;border-style:solid;border-width:1px;font-family:Arial, sans-serif;font-size:14px;
  font-weight:normal;overflow:hidden;padding:10px 5px;word-break:normal;}
.tg .tg-za14{border-color:inherit;text-align:left;vertical-align:bottom}
.tg .tg-0pky{border-color:inherit;text-align:left;vertical-align:top}
.tg .tg-7zrl{text-align:left;vertical-align:bottom}
.tg .tg-0lax{text-align:left;vertical-align:top}
</style>
<table class="tg"><thead>
  <tr>
    <th class="tg-za14">Feature</th>
    <th class="tg-za14">NLTK (Natural Language Toolkit)</th>
    <th class="tg-za14">spaCy</th>
  </tr></thead>
<tbody>
  <tr>
    <td class="tg-za14">Primary Goal</td>
    <td class="tg-za14">Education and Research</td>
    <td class="tg-0pky">Production and Performance</td>
  </tr>
  <tr>
    <td class="tg-za14">Logic Type</td>
    <td class="tg-za14">Rule-based and Statistical "DIY"</td>
    <td class="tg-0pky">Neural Network / Transformer-based</td>
  </tr>
  <tr>
    <td class="tg-za14">Ease of Use</td>
    <td class="tg-za14">Manual (Requires manual POS tagging)</td>
    <td class="tg-0pky">Automatic (One-line execution)</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Execution Speed</td>
    <td class="tg-7zrl">Slower (Python-heavy)</td>
    <td class="tg-0lax">Very Fast (Optimized Cython/C++)</td>
  </tr>
  <tr>
    <td class="tg-7zrl">NER Accuracy</td>
    <td class="tg-7zrl">Moderate (Uses older classifiers)</td>
    <td class="tg-0lax">High (State-of-the-art models)</td>
  </tr>
  <tr>
    <td class="tg-7zrl">Visualization</td>
    <td class="tg-7zrl">Text-based tree structures</td>
    <td class="tg-0lax">displaCy (Web-based interactive visuals)</td>
  </tr>
</tbody></table>

**Comparision Implementation**

Creates a side-by-side text table comparing entities detected by NLTK and spaCy on the exact same 3 abstracts. Clearly shows differences in quantity, label types, precision, and how each system handles technical terms, acronyms, numbers, etc.

In [ ]:
sample_indices = df_sample.head(3).index

print(f"{'='*80}")
print(f"NER COMPARISON: NLTK vs. SPACY")
print(f"{'='*80}\n")

for i, idx in enumerate(sample_indices, 1):
    full_text = df.loc[idx]
    
    nltk_ents = df_entities.loc[idx]
    
    doc = nlp(full_text)
    spacy_ents = [(ent.text, ent.label_) for ent in doc.ents]

    print(f"ROW {i} (Index: {idx})")
    print(f"PARAGRAPH:\n{full_text}\n")

    print(f"{'NLTK ENTITIES':<35} | {'SPACY ENTITIES':<35}")
    print(f"{'-'*35} | {'-'*35}")
    
    max_len = max(len(nltk_ents), len(spacy_ents))
    
    for n in range(max_len):
        n_str = f"{nltk_ents[n][0]} [{nltk_ents[n][1]}]" if n < len(nltk_ents) else ""
        s_str = f"{spacy_ents[n][0]} [{spacy_ents[n][1]}]" if n < len(spacy_ents) else ""
        print(f"{n_str:<35} | {s_str:<35}")
    
    print(f"\n{'='*80}\n")

NER COMPARISON: NLTK vs. SPACY

ROW 1 (Index: 29096)
PARAGRAPH:
Monocular depth prediction plays a crucial role in understanding 3D scene
geometry. Although recent methods have achieved impressive progress in
evaluation metrics such as the pixel-wise relative error, most methods neglect
the geometric constraints in the 3D space. In this work, we show the importance
of the high-order 3D geometric constraints for depth prediction. By designing a
loss term that enforces one simple type of geometric constraints, namely,
virtual normal directions determined by randomly sampled three points in the
reconstructed 3D space, we can considerably improve the depth prediction
accuracy. Significantly, the byproduct of this predicted depth being
sufficiently accurate is that we are now able to recover good 3D structures of
the scene such as the point cloud and surface normal directly from the depth,
eliminating the necessity of training new sub-models as was previously done.
Experiments on two benchm

# Conclusion

This notebook successfully demonstrates and compares two popular NER implementations on real arXiv abstracts. NLTK provides a fast, lightweight baseline that reliably detects some organizations and locations but frequently over-labels technical terms and acronyms as ORGANIZATION. spaCy, with its more modern statistical model, delivers richer and generally more accurate entity types (especially CARDINAL numbers, PRODUCT names like model acronyms, and better distinction of GPE vs ORG), making it superior for technical/academic text. The displaCy visualizations make the differences immediately interpretable and highlight the syntactic environment around entities. Overall, the experiment shows that while classic NLTK NER is still useful for quick prototyping, spaCy is the better choice for production-quality entity extraction in scientific domains. This side-by-side analysis serves as a valuable educational resource for understanding NER system behavior and trade-offs.

# References

- [spaCy](https://spacy.io/usage/linguistic-features#named-entities)
- [Wikipedia Information Extraction](https://en.wikipedia.org/wiki/Information_extraction)
- [NER Geeksforgeeks](https://www.geeksforgeeks.org/nlp/named-entity-recognition/)